# Inteligência Artificial em Jogos 25/26
## Projeto NEAT - Versão 2: Navegação por Waypoints

**Introdução**
Este notebook implementa a abordagem de navegação guiada exigida no **Requisito 3.1** do projeto. Nesta variante, o carro não utiliza sensores de colisão para detetar os limites físicos da pista. Em vez disso, a rede neuronal é informada da localização do próximo "ponto-chave" (Waypoint) que deve alcançar.

*(Nota: Toda a estrutura visual e motor de física encontra-se no ficheiro auxiliar `base_jogo.py`).*

In [2]:
from base_jogo import *

pygame 2.6.1 (SDL 2.28.4, Python 3.12.4)
Hello from the pygame community. https://www.pygame.org/contribute.html


### 1. Classe NeatCar e Dinâmica de Navegação

Para esta versão, o cérebro do nosso carro recebe inputs estritamente relativos ao seu referencial face ao objetivo:
* **Input 1:** A distância euclidiana normalizada entre o centro do carro e o waypoint alvo.
* **Input 2:** A diferença angular (normalizada de -1 a 1) entre a rotação atual do carro e o ângulo necessário para apontar diretamente para o waypoint.

À semelhança da versão dos radares, e de forma a cumprir o **Requisito 3.3**, os outputs (*throttle* e *steering*) sofrem a injeção de ruído para simular imperfeições no sistema de tração/direção.

In [3]:
class NeatCar(AbstractCar):
    IMG = GREEN_CAR
    START_POS = (150, 200)

    def __init__(self, max_vel, rotation_vel, start_angle=0, track_reversed=False):
        super().__init__(max_vel, rotation_vel)
        self.angle = start_angle
        self.x, self.y = (150, 200)
        self.alive = True
        self.distance = 0
        self.current_waypoint = 0
        self.finished = False
        self.finish_time = 0

    def get_waypoint_data(self, waypoints):
        if len(waypoints) == 0:
            return [0, 0]
        
        target_x, target_y = waypoints[self.current_waypoint]
        
        # Input 1: Distância normalizada
        dist = math.hypot(target_x - self.x, target_y - self.y)
        dist_normalizada = dist / math.hypot(WIDTH, HEIGHT)
        
        # Input 2: Ângulo relativo ao próximo waypoint
        x_diff = target_x - self.x
        y_diff = target_y - self.y
        if y_diff == 0:
            desired_angle = (1 if x_diff < 0 else -1) * 90
        else:
            desired_angle = math.degrees(math.atan(x_diff / y_diff))
        if target_y > self.y:
            desired_angle += 180
        angle_diff = (self.angle - desired_angle) % 360
        if angle_diff > 180:
            angle_diff -= 360
        angle_normalizado = -angle_diff / 180

        return [dist_normalizada, angle_normalizado]

    def update_waypoint(self, waypoints):
        target_x, target_y = waypoints[self.current_waypoint]
        dist = math.hypot(target_x - self.x, target_y - self.y)
        if dist < 25:
            self.current_waypoint += 1
            if self.current_waypoint >= len(waypoints):
                self.current_waypoint = 0

    def apply_nn_actions(self, throttle, steering):
        noise_throttle = uniform(-0.05, 0.05)
        noise_steering = uniform(-0.05, 0.05)
        throttle = max(-1.0, min(1.0, throttle + noise_throttle))
        steering = max(-1.0, min(1.0, steering + noise_steering))
        if throttle > 0:
            self.vel = min(self.vel + self.acceleration * throttle, self.max_vel)
        else:
            self.vel = max(self.vel + (self.acceleration * 2) * throttle, 0)
        self.angle += self.rotation_vel * steering
        self.angle = int(self.angle) % 360
        self.move()
        self.distance += self.vel

    def check_collision(self, frame_count):
        if self.collide(TRACK_BORDER_MASK) != None:
            self.alive = False
            self.vel = 0
        if frame_count > 180:
            finish_poi_collide = self.collide(FINISH_MASK, *FINISH_POSITION)
            if finish_poi_collide != None:
                if finish_poi_collide[1] == 0:
                    self.alive = False
                    self.vel = 0
                else:
                    self.finished = True
                    self.vel = 0
                    self.finish_time = frame_count / FPS

    def draw(self, win):
        super().draw(win)

### 2. Definição da Trajetória (Waypoints)
O código abaixo permite inicializar a lista de waypoints do circuito. O utilizador pode desenhar os pontos de referência em tempo real, os quais definirão a "linha de corrida ideal" (o cume das curvas e as tangentes) para onde o agente inteligente se deve dirigir.

In [4]:
pygame.init()
global WIN
WIN = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Marca os Waypoints!")

waypoints = []
run = True
clock = pygame.time.Clock()

print("Clica na pista para adicionar waypoints. Prime ENTER para terminar.")

while run:
    clock.tick(FPS)
    
    WIN.blit(GRASS, (0, 0))
    WIN.blit(TRACK, (0, 0))

    for i, point in enumerate(waypoints):
        pygame.draw.circle(WIN, (255, 0, 0), point, 6)
        if i > 0:
            pygame.draw.line(WIN, (255, 0, 0), waypoints[i-1], point, 2)

    pygame.display.update()

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            run = False
            pygame.quit()
            sys.exit()

        if event.type == pygame.MOUSEBUTTONDOWN:
            pos = pygame.mouse.get_pos()
            waypoints.append(pos)
            print(f"Waypoint {len(waypoints)}: {pos}")

        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_RETURN:
                run = False

print(f"\nWaypoints guardados: {waypoints}")

Clica na pista para adicionar waypoints. Prime ENTER para terminar.
Waypoint 1: (123, 73)
Waypoint 2: (93, 84)
Waypoint 3: (81, 94)
Waypoint 4: (61, 117)
Waypoint 5: (56, 145)
Waypoint 6: (55, 187)
Waypoint 7: (55, 261)
Waypoint 8: (53, 352)
Waypoint 9: (54, 441)
Waypoint 10: (80, 506)
Waypoint 11: (129, 548)
Waypoint 12: (172, 611)
Waypoint 13: (216, 647)
Waypoint 14: (239, 674)
Waypoint 15: (293, 721)
Waypoint 16: (329, 737)
Waypoint 17: (363, 733)
Waypoint 18: (373, 728)
Waypoint 19: (394, 719)
Waypoint 20: (404, 697)
Waypoint 21: (409, 672)
Waypoint 22: (414, 641)
Waypoint 23: (416, 605)
Waypoint 24: (416, 579)
Waypoint 25: (423, 548)
Waypoint 26: (427, 517)
Waypoint 27: (449, 498)
Waypoint 28: (475, 491)
Waypoint 29: (511, 488)
Waypoint 30: (534, 488)
Waypoint 31: (561, 508)
Waypoint 32: (569, 545)
Waypoint 33: (577, 568)
Waypoint 34: (579, 603)
Waypoint 35: (585, 627)
Waypoint 36: (592, 656)
Waypoint 37: (599, 687)
Waypoint 38: (611, 704)
Waypoint 39: (636, 712)
Waypoint 40: (667

### 3. Função de Fitness (Waypoints)

Para garantir que o carro aprende a seguir a pista em vez de apenas andar às voltas, a função de avaliação (`eval_genomes_waypoints`) altera a forma como o *fitness* é atribuído (cumprindo o **Requisito 4**):

* O carro "recolhe" um waypoint se se aproximar num raio de tolerância (ex: 25 píxeis).
* **Recompensa por Meta:** O atingimento de um novo waypoint confere um *bónus de fitness* imediato de 200 pontos.
* Carros lentos no arranque ou que choquem com as bordas da pista perdem imediatamente a corrida e os respetivos cérebros sofrem penalizações indiretas ao serem removidos prematuramente do cálculo base de velocidade.

In [ ]:
def eval_genomes_waypoints(genomes, config):
    INVERTER_PISTA = False
    ANGULO_INICIAL = 180 if INVERTER_PISTA else 0

    nets = []
    cars = []
    ge = []

    for genome_id, genome in genomes:
        genome.fitness = 0
        net = neat.nn.FeedForwardNetwork.create(genome, config)
        nets.append(net)
        cars.append(NeatCar(max_vel=4, rotation_vel=5.5, start_angle=ANGULO_INICIAL, track_reversed=INVERTER_PISTA))
        ge.append(genome)

    clock = pygame.time.Clock()
    frame_count = 0

    selected_waypoint = None
    while len(cars) > 0:
        clock.tick(FPS)
        frame_count += 1

        if frame_count > 20000:
            break

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()
            
            if event.type == pygame.MOUSEBUTTONDOWN:
                mouse_x, mouse_y = pygame.mouse.get_pos()
                for j, point in enumerate(waypoints):
                    if math.hypot(mouse_x - point[0], mouse_y - point[1]) < 15:
                        selected_waypoint = j
                        break
            
            if event.type == pygame.MOUSEBUTTONUP:
                selected_waypoint = None
            
            if event.type == pygame.MOUSEMOTION and selected_waypoint is not None:
                waypoints[selected_waypoint] = pygame.mouse.get_pos()

        WIN.blit(GRASS, (0, 0))
        WIN.blit(TRACK, (0, 0))
        WIN.blit(FINISH, FINISH_POSITION)

        for point in waypoints:
            pygame.draw.circle(WIN, (255, 0, 0), point, 5)

        for i in reversed(range(len(cars))):
            car = cars[i]

            old_waypoint = car.current_waypoint
            car.update_waypoint(waypoints)
            inputs = car.get_waypoint_data(waypoints)
            output = nets[i].activate(inputs)

            throttle = output[0]
            steering = output[1]

            if abs(steering) < 0.2:
                steering = 0

            car.apply_nn_actions(throttle, steering)
            car.check_collision(frame_count)

            if frame_count == 180:
                dist_start = math.hypot(car.x - 150, car.y - 200)
                if dist_start < 100:
                    car.alive = False

            if car.finished:
                ge[i].fitness += 100000
                print(f"\n[!] SUCESSO! O carro encontrou a meta em {car.finish_time:.2f} segundos!")
                
                with open('models/waypoints_finais.pkl', 'wb') as f:
                    pickle.dump(waypoints, f)
                print("Waypoints exatos guardados no ficheiro 'waypoints_finais.pkl'")
                
                cars.pop(i)
                nets.pop(i)
                ge.pop(i)
                break

            elif not car.alive or (car.vel <= 0 and frame_count > 60):
                cars.pop(i)
                nets.pop(i)
                ge.pop(i)

            else:
                if car.current_waypoint != old_waypoint:
                    ge[i].fitness += 200

                car.draw(WIN)
                wp_font = pygame.font.SysFont("comicsans", 20)
                wp_txt = wp_font.render(str(car.current_waypoint), 1, (255, 255, 0))
                WIN.blit(wp_txt, (int(car.x), int(car.y)))

        pygame.display.update()

### 4. Execução do Algoritmo

A célula abaixo inicializa a população e executa a evolução baseada nos pesos do ficheiro de configuração feedforward correspondente. Uma pista de migalhas (*breadcrumbs*) foi adicionada graficamente à simulação para observação visual clara do waypoint alvo ativo.
No final, os dados são exportados via `pickle` para análise posterior no relatório.

In [ ]:
def run_neat_waypoints(config_path):
    pygame.init()
    global WIN
    WIN = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption("Racing Game - Waypoints!")

    config = neat.config.Config(neat.DefaultGenome, neat.DefaultReproduction,
                                neat.DefaultSpeciesSet, neat.DefaultStagnation,
                                config_path)

    p = neat.Population(config)

    p.add_reporter(neat.StdOutReporter(True))
    stats = neat.StatisticsReporter()
    p.add_reporter(stats)

    import time
    start_time = time.time()
    winner = p.run(eval_genomes_waypoints, 25)
    elapsed_time = time.time() - start_time

    pygame.quit()

    try:
        import pickle
        with open('models/winner_waypoints.pkl', 'wb') as f:
            pickle.dump(winner, f)

        info = {
            'melhor_fitness': winner.fitness,
            'melhor_geracao': p.generation,
            'tempo_total': elapsed_time,
            'num_waypoints': len(waypoints),
            'config': config_path,
            'historico_melhor': [c.fitness for c in stats.most_fit_genomes],
            'historico_media': stats.get_fitness_mean()
        }
        with open('models/stats_waypoints.pkl', 'wb') as f:
            pickle.dump(info, f)

        print(f"\n[OK] Ficheiros guardados com sucesso!")
        print(f"Melhor fitness: {winner.fitness}")
        print(f"Melhor geração: {p.generation}")
        print(f"Tempo total: {elapsed_time:.1f} segundos")
        print(f"Número de waypoints: {len(waypoints)}")
        
    except Exception as e:
        print(f"Erro ao guardar: {e}")

    print('\nMelhor genoma encontrado:\n{!s}'.format(winner))

# Corre novamente o treino
run_neat_waypoints('configs/config-waypoints.txt')


 ****** Running generation 0 ****** 

Population's average fitness: 1.66667 stdev: 18.18119
Best fitness: 200.00000 - size: (2, 4) - species 1 - id 1
Average adjusted fitness: 0.008
Mean genetic distance 2.500, standard deviation 0.943
Population of 120 members in 3 species (after reproduction):
   ID   age  size   fitness   adj fit  stag
  ====  ===  ====  =========  =======  ====
     1    0   106    200.000    0.008     0
     2    0    12         --       --     0
     3    0     2         --       --     0
Total extinctions: 0
Generation time: 1.979 sec

 ****** Running generation 1 ****** 

Population's average fitness: 25.00000 stdev: 204.22618
Best fitness: 2200.00000 - size: (3, 5) - species 1 - id 237
Average adjusted fitness: 0.004
Mean genetic distance 2.489, standard deviation 0.763
Population of 120 members in 3 species (after reproduction):
   ID   age  size   fitness   adj fit  stag
  ====  ===  ====  =========  =======  ====
     1    1    95   2200.000    0.013     0